In [1]:
import os
import json
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

In [2]:
BASE_DIR = 'PartImageNet_Seg/PartImageNet'
ANNOTATIONS_DIR = os.path.join(BASE_DIR, 'annotations')
IMAGES_DIR = os.path.join(BASE_DIR, 'images')

# --- Training Configuration ---
# You can choose which dataset to use in the main function
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4

In [3]:
# TARGET_CLASSES = [
#     'n02691156',  # airplane
#     'n02131653',  # bear
#     'n02834778',  # bicycle
#     'n01503061',  # bird
#     'n02858304',  # boat
#     'n02876657',  # bottle
#     'n02958343',  # car
#     'n02121808',  # cat
#     'n03001627',  # chair
#     'n03046257',  # clock
#     'n02084071',  # dog
#     'n02503517',  # elephant
#     'n03627232',  # keyboard
#     'n03633091',  # knife
#     'n03874599',  # oven
#     'n04468005'   # truck
# ]

NUM_PARTS = 10 # The value you found using the analysis script

# The total number of channels will be the parts + 1 for the global image
TOTAL_CHANNELS = NUM_PARTS + 1

## Simple CNN with independent channel

In [4]:
class PartImageNetCOCODataset(Dataset):
    """
    Loads data from a single COCO JSON, and correctly constructs the path to images
    that are directly inside the split folder (e.g., 'images/train/').
    """
    def __init__(self, coco_json_path, image_split_dir, num_parts, img_dims):
        self.coco_json_path = coco_json_path
        self.image_split_dir = image_split_dir # The specific split dir, e.g., '.../images/train'
        self.num_parts = num_parts
        self.img_height, self.img_width = img_dims
        self.data = self._load_coco_data()
        self.image_ids = list(self.data['images'].keys())

    def _load_coco_data(self):
        if not os.path.exists(self.coco_json_path):
            raise FileNotFoundError(f"❌ COCO JSON file not found at: {self.coco_json_path}")
        print(f"Loading annotations from: {self.coco_json_path}...")
        with open(self.coco_json_path, 'r') as f:
            raw_data = json.load(f)
        images = {img['id']: img for img in raw_data['images']}
        annotations_by_image = {img_id: [] for img_id in images}
        for ann in raw_data['annotations']:
            if ann['image_id'] in annotations_by_image:
                annotations_by_image[ann['image_id']].append(ann)
        print(f"✅ Loaded {len(images)} images and {len(raw_data['annotations'])} annotations.")
        return {'images': images, 'annotations': annotations_by_image, 'categories': raw_data.get('categories', [])}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.data['images'][image_id]
        annotations = self.data['annotations'].get(image_id, [])
        
        # --- FINAL FIX: Construct path without the extra subfolder ---
        image_filename_from_json = image_info['file_name']
        # The file might be in a subfolder in the JSON, but not on disk.
        # We take only the final part of the path (the actual filename).
        base_filename = os.path.basename(image_filename_from_json)
        
        # Ensure the extension is correct (.JPEG)
        base_name_no_ext, _ = os.path.splitext(base_filename)
        correct_filename = base_name_no_ext + '.JPEG'
        
        # The full path is the specific split directory + the corrected filename.
        image_path = os.path.join(self.image_split_dir, correct_filename)
        
        global_image = self._load_global_image(image_path)
        if global_image is None: 
            return None

        part_masks = np.zeros((self.num_parts, self.img_height, self.img_width), dtype=np.float32)
        label = -1
        if annotations:
            for ann in annotations:
                part_id = ann.get('part_id')
                if part_id is not None and part_id < self.num_parts:
                    x, y, w, h = ann['bbox']
                    x_start = int((x / image_info['width']) * self.img_width)
                    y_start = int((y / image_info['height']) * self.img_height)
                    x_end = int(((x + w) / image_info['width']) * self.img_width)
                    y_end = int(((y + h) / image_info['height']) * self.img_height)
                    part_masks[part_id, y_start:y_end, x_start:x_end] = 1.0
            if annotations:
                label = int(annotations[0]['category_id'])

        combined_tensor = torch.from_numpy(np.vstack([part_masks, global_image]))
        return combined_tensor, torch.tensor(label, dtype=torch.long)

    def _load_global_image(self, image_path):
        try:
            with Image.open(image_path).convert('RGB').convert('L') as img:
                img = img.resize((self.img_width, self.img_height), Image.Resampling.LANCZOS)
            global_image = np.array(img, dtype=np.float32) / 255.0
            return np.expand_dims(global_image, axis=0)
        except FileNotFoundError:
            # This is expected for some files. The collate_fn will handle the `None` return.
            # print(f"INFO: Skipping missing image file: {image_path}")
            return None 
        except Exception as e:
            # print(f"WARNING: Could not load image {image_path}. Reason: {e}")
            return None

In [5]:
def collate_fn_skip_errors(batch):
    """A custom collate function that filters out None values from a batch."""
    # This can be useful if __getitem__ returns None on an error
    batch = list(filter(lambda x: x is not None, batch))
    if not batch:
        return torch.tensor([]), torch.tensor([])
    return torch.utils.data.dataloader.default_collate(batch)

In [6]:
class PartMaskCNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(PartMaskCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), nn.Dropout(0.25),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier_input_size = 128 * (IMG_HEIGHT // 8) * (IMG_WIDTH // 8)
        self.classifier = nn.Sequential(
            nn.Linear(self.classifier_input_size, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.features(x); x = x.view(-1, self.classifier_input_size); x = self.classifier(x); return x

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_json_path = os.path.join(ANNOTATIONS_DIR, 'train_whole', 'train.json')
with open(train_json_path, 'r') as f:
    train_data = json.load(f)
NUM_CLASSES = len(train_data['categories']) + 1
print(f"Determined {NUM_CLASSES} classes from train.json.")

print("\nLoading datasets...")
train_dataset = PartImageNetCOCODataset(
    coco_json_path=train_json_path,
    image_split_dir=os.path.join(IMAGES_DIR, 'train'), 
    num_parts=NUM_PARTS, img_dims=(IMG_HEIGHT, IMG_WIDTH)
)
val_dataset = PartImageNetCOCODataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'val_whole', 'val.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'val'),
    num_parts=NUM_PARTS, img_dims=(IMG_HEIGHT, IMG_WIDTH)
)

if len(train_dataset) == 0:
    print("\n❌ FATAL: Training dataset is empty.")
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, collate_fn=collate_fn_skip_errors)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=collate_fn_skip_errors)

    model = PartMaskCNN(in_channels=TOTAL_CHANNELS, num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("\nModel Architecture:"); print(model)

Using device: cuda
Determined 159 classes from train.json.

Loading datasets...
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/train_whole/train.json...
✅ Loaded 20481 images and 20457 annotations.
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/val_whole/val.json...
✅ Loaded 1206 images and 1205 annotations.

Model Architecture:
PartMaskCNN(
  (features): Sequential(
    (0): Conv2d(11, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Dropout(p=0.25, inplace=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.25, inplace=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_siz

In [22]:
print("\n--- Starting Training ---")

for epoch in range(EPOCHS):
    model.train(); running_loss = 0.0; train_correct = 0; train_total = 0
    for i, (inputs, labels) in enumerate(train_loader):
        if inputs.nelement() == 0: continue
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0); train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total if train_total > 0 else 0
    avg_train_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0

    model.eval(); val_correct = 0; val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            if inputs.nelement() == 0: continue
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0); val_correct += (predicted == labels).sum().item()
    
    val_accuracy = 100 * val_correct / val_total if val_total > 0 else 0

    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Acc: {val_accuracy:.2f}%")

print("\n✅ Training finished successfully!")


--- Starting Training ---


KeyboardInterrupt: 

## Independent Channel with ResNet

In [35]:
from torchvision.models import resnet18, ResNet18_Weights

In [36]:
class PartImageNetCOCODataset(Dataset):
    def __init__(self, coco_json_path, image_split_dir, img_dims, transforms=None):
        self.coco_json_path = coco_json_path
        self.image_split_dir = image_split_dir
        self.img_height, self.img_width = img_dims
        self.transforms = transforms # Store the transforms
        self.data, self.max_part_id, self.max_category_id = self._load_coco_data()
        self.num_parts = self.max_part_id + 1
        self.image_ids = list(self.data['images'].keys())

    def _load_coco_data(self):
        if not os.path.exists(self.coco_json_path):
            raise FileNotFoundError(f"❌ COCO JSON file not found at: {self.coco_json_path}")
        print(f"Loading annotations from: {self.coco_json_path}...")
        with open(self.coco_json_path, 'r') as f:
            raw_data = json.load(f)
        max_part_id = 0
        max_category_id = 0
        for ann in raw_data['annotations']:
            if ann.get('part_id', 0) > max_part_id:
                max_part_id = ann['part_id']
            if ann.get('category_id', 0) > max_category_id:
                max_category_id = ann['category_id']
        images = {img['id']: img for img in raw_data['images']}
        annotations_by_image = {img_id: [] for img_id in images}
        for ann in raw_data['annotations']:
            if ann['image_id'] in annotations_by_image:
                annotations_by_image[ann['image_id']].append(ann)
        print(f"✅ Loaded {len(images)} images. Max Part ID: {max_part_id}, Max Category ID: {max_category_id}")
        data = {'images': images, 'annotations': annotations_by_image}
        return data, max_part_id, max_category_id

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.data['images'][image_id]
        annotations = self.data['annotations'].get(image_id, [])
        
        image_filename_from_json = os.path.basename(image_info['file_name'])
        base_name, _ = os.path.splitext(image_filename_from_json)
        correct_filename = base_name + '.JPEG'
        image_path = os.path.join(self.image_split_dir, correct_filename)
        
        try:
            img = Image.open(image_path).convert('RGB')
            img = img.resize((self.img_width, self.img_height), Image.Resampling.LANCZOS)
        except FileNotFoundError:
            return None 

        part_masks = np.zeros((self.num_parts, self.img_height, self.img_width), dtype=np.float32)
        label = -1
        if annotations:
            for ann in annotations:
                part_id = ann.get('part_id')
                if part_id is not None:
                    x, y, w, h = ann['bbox']
                    x_start = int((x / image_info['width']) * self.img_width)
                    y_start = int((y / image_info['height']) * self.img_height)
                    x_end = int(((x + w) / image_info['width']) * self.img_width)
                    y_end = int(((y + h) / image_info['height']) * self.img_height)
                    part_masks[part_id, y_start:y_end, x_start:x_end] = 1.0
            label = int(annotations[0]['category_id'])

        img_tensor = T.ToTensor()(img)
        if self.transforms:
            img_tensor = self.transforms(img_tensor)

        part_masks_tensor = torch.from_numpy(part_masks)
        combined_tensor = torch.cat([part_masks_tensor, img_tensor], dim=0)
        
        return combined_tensor, torch.tensor(label, dtype=torch.long)

# --- Model Architecture ---
def create_adapted_resnet(in_channels, num_classes):
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    original_conv1 = model.conv1
    original_weights = original_conv1.weight.clone()
    new_conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    with torch.no_grad():
        # Place the original RGB weights at the end of the new weight tensor
        new_conv1.weight[:, -3:, :, :] = original_weights 
        # Initialize the new part channels' weights to zero
        new_conv1.weight[:, :-3, :, :] = 0.0
    model.conv1 = new_conv1
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model

def collate_fn_skip_errors(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if not batch: return torch.tensor([]), torch.tensor([])
    return torch.utils.data.dataloader.default_collate(batch)

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. Define Data Augmentation Pipeline ---
train_transforms = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    T.RandomRotation(15),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard for ImageNet
])

val_transforms = T.Compose([
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 2. Load Datasets and Apply Transforms ---
print("\nLoading datasets...")
train_dataset = PartImageNetCOCODataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'train_whole', 'train.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'train'), 
    img_dims=(IMG_HEIGHT, IMG_WIDTH),
    transforms=train_transforms # Apply augmentation to training set
)
val_dataset = PartImageNetCOCODataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'val_whole', 'val.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'val'),
    img_dims=(IMG_HEIGHT, IMG_WIDTH),
    transforms=val_transforms # Only apply normalization to validation set
)

NUM_PARTS = train_dataset.num_parts
TOTAL_CHANNELS = NUM_PARTS + 3
NUM_CLASSES = max(train_dataset.max_category_id, val_dataset.max_category_id) + 1
print(f"Determined NUM_PARTS={NUM_PARTS} and NUM_CLASSES={NUM_CLASSES} from the data.")

if len(train_dataset) == 0:
    print("\n❌ FATAL: Training dataset is empty.")
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, collate_fn=collate_fn_skip_errors)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=collate_fn_skip_errors)

    model = create_adapted_resnet(in_channels=TOTAL_CHANNELS, num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # --- 3. Early Stopping Setup ---
    best_val_accuracy = 0.0
    epochs_no_improve = 0
    patience = 5 # Number of epochs to wait for improvement before stopping

print("\nModel Architecture: Adapted ResNet18"); print(model)

Using device: cuda

Loading datasets...
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/train_whole/train.json...
✅ Loaded 20481 images. Max Part ID: 0, Max Category ID: 157
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/val_whole/val.json...
✅ Loaded 1206 images. Max Part ID: 0, Max Category ID: 157
Determined NUM_PARTS=1 and NUM_CLASSES=158 from the data.

Model Architecture: Adapted ResNet18
ResNet(
  (conv1): Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=T

In [ ]:
print("\n--- Starting Training with Augmentation and Early Stopping ---")

for epoch in range(EPOCHS):
    model.train(); running_loss = 0.0; train_correct = 0; train_total = 0
    for i, (inputs, labels) in enumerate(train_loader):
        if inputs.nelement() == 0: continue
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0); train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total if train_total > 0 else 0
    avg_train_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0

    model.eval(); val_correct = 0; val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            if inputs.nelement() == 0: continue
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0); val_correct += (predicted == labels).sum().item()
    
    val_accuracy = 100 * val_correct / val_total if val_total > 0 else 0

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.2f}% | Val Acc: {val_accuracy:.2f}%")

    # --- 4. Early Stopping Check ---
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_independent_channel_model.pth')
        print(f"  -> New best validation accuracy: {best_val_accuracy:.2f}%. Model saved.")
    else:
        epochs_no_improve += 1
    
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered after {patience} epochs with no improvement.")
        break

print("\n✅ Training finished.")


--- Starting Training ---


KeyboardInterrupt: 

In [ ]:
save_path = './ind_resnet.pth'
        
# Save the model's state_dict
torch.save(model.state_dict(), save_path)

print(f"✅ Model saved to {save_path}")

## Shared Channel with ResNet

In [27]:
import torchvision.transforms as T

In [26]:
class PartImageNetCOCODataset(Dataset):
    def __init__(self, coco_json_path, image_split_dir, num_parts, img_dims, transforms=None):
        self.coco_json_path = coco_json_path
        self.image_split_dir = image_split_dir
        self.num_parts = num_parts
        self.img_height, self.img_width = img_dims
        self.transforms = transforms # Store the transforms
        self.data, self.max_part_id, self.max_category_id = self._load_coco_data()
        self.image_ids = list(self.data['images'].keys())

    def _load_coco_data(self):
        # ... (This function remains the same)
        if not os.path.exists(self.coco_json_path):
            raise FileNotFoundError(f"❌ COCO JSON file not found at: {self.coco_json_path}")
        print(f"Loading annotations from: {self.coco_json_path}...")
        with open(self.coco_json_path, 'r') as f:
            raw_data = json.load(f)
        max_part_id = 0
        max_category_id = 0
        for ann in raw_data['annotations']:
            if ann.get('part_id', 0) > max_part_id:
                max_part_id = ann['part_id']
            if ann.get('category_id', 0) > max_category_id:
                max_category_id = ann['category_id']
        images = {img['id']: img for img in raw_data['images']}
        annotations_by_image = {img_id: [] for img_id in images}
        for ann in raw_data['annotations']:
            if ann['image_id'] in annotations_by_image:
                annotations_by_image[ann['image_id']].append(ann)
        print(f"✅ Loaded {len(images)} images. Max Part ID: {max_part_id}, Max Category ID: {max_category_id}")
        data = {'images': images, 'annotations': annotations_by_image}
        return data, max_part_id, max_category_id


    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # ... (The first part of this function remains the same)
        image_id = self.image_ids[idx]
        image_info = self.data['images'][image_id]
        annotations = self.data['annotations'].get(image_id, [])
        image_filename_from_json = image_info['file_name']
        base_name, _ = os.path.splitext(image_filename_from_json)
        correct_filename = base_name + '.JPEG'
        image_path = os.path.join(self.image_split_dir, correct_filename)
        
        # --- CHANGE: Load image but do not convert to numpy yet ---
        try:
            img = Image.open(image_path).convert('RGB')
            img = img.resize((self.img_width, self.img_height), Image.Resampling.LANCZOS)
        except FileNotFoundError:
            return None 

        part_masks = np.zeros((self.num_parts, self.img_height, self.img_width), dtype=np.float32)
        label = -1
        if annotations:
            for ann in annotations:
                part_id = ann.get('part_id')
                if part_id is not None:
                    x, y, w, h = ann['bbox']
                    x_start = int((x / image_info['width']) * self.img_width)
                    y_start = int((y / image_info['height']) * self.img_height)
                    x_end = int(((x + w) / image_info['width']) * self.img_width)
                    y_end = int(((y + h) / image_info['height']) * self.img_height)
                    part_masks[part_id, y_start:y_end, x_start:x_end] = 1.0
            label = int(annotations[0]['category_id'])

        # --- CHANGE: Apply transforms to the image ---
        img_tensor = T.ToTensor()(img) # Converts to (C, H, W) and scales to [0, 1]
        
        # Apply other transforms if they exist
        if self.transforms:
            img_tensor = self.transforms(img_tensor)

        part_masks_tensor = torch.from_numpy(part_masks)
        combined_tensor = torch.cat([part_masks_tensor, img_tensor], dim=0)
        
        return combined_tensor, torch.tensor(label, dtype=torch.long)

In [11]:
def create_adapted_resnet(in_channels, num_classes):
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    original_conv1 = model.conv1
    original_weights = original_conv1.weight.clone()
    new_conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    with torch.no_grad():
        # The RGB channels are now at the end of our input tensor
        new_conv1.weight[:, -3:, :, :] = original_weights 
        new_conv1.weight[:, :-3, :, :] = 0.0
    model.conv1 = new_conv1
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model

def collate_fn_skip_errors(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if not batch: return torch.tensor([]), torch.tensor([])
    return torch.utils.data.dataloader.default_collate(batch)

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. Define Data Augmentation Pipeline ---
# These transforms will be applied to the training set
train_transforms = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.RandomRotation(10),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard for ImageNet
])

# Validation set should only have normalization, no augmentation
val_transforms = T.Compose([
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Load Datasets and Apply Transforms
train_dataset = PartImageNetCOCODataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'train_whole', 'train.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'train'), 
    num_parts=NUM_PARTS, 
    img_dims=(IMG_HEIGHT, IMG_WIDTH),
    transforms=train_transforms # Apply augmentation
)
val_dataset = PartImageNetCOCODataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'val_whole', 'val.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'val'),
    num_parts=NUM_PARTS, 
    img_dims=(IMG_HEIGHT, IMG_WIDTH),
    transforms=val_transforms # Only apply normalization
)

NUM_PARTS = train_dataset.num_parts
TOTAL_CHANNELS = NUM_PARTS + 3
NUM_CLASSES = max(train_dataset.max_category_id, val_dataset.max_category_id) + 1

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, collate_fn=collate_fn_skip_errors)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=collate_fn_skip_errors)

model = create_adapted_resnet(in_channels=TOTAL_CHANNELS, num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- 3. Early Stopping Setup ---
best_val_accuracy = 0
epochs_no_improve = 0
patience = 5 # Number of epochs to wait for improvement before stopping

print("\nModel Architecture:"); print(model)

Using device: cuda
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/train_whole/train.json...
✅ Loaded 20481 images. Max Part ID: 0, Max Category ID: 157
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/val_whole/val.json...
✅ Loaded 1206 images. Max Part ID: 0, Max Category ID: 157

Model Architecture:
ResNet(
  (conv1): Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False

In [29]:
print("\n--- Starting Training with Augmentation and Early Stopping ---")

for epoch in range(EPOCHS):
    # ... (Training loop remains the same)
    model.train(); running_loss = 0.0; train_correct = 0; train_total = 0
    for i, (inputs, labels) in enumerate(train_loader):
        if inputs.nelement() == 0: continue
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0); train_correct += (predicted == labels).sum().item()
    train_accuracy = 100 * train_correct / train_total if train_total > 0 else 0
    avg_train_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0

    # ... (Validation loop remains the same)
    model.eval(); val_correct = 0; val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            if inputs.nelement() == 0: continue
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0); val_correct += (predicted == labels).sum().item()
    val_accuracy = 100 * val_correct / val_total if val_total > 0 else 0

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.2f}% | Val Acc: {val_accuracy:.2f}%")

    # --- 4. Early Stopping Check ---
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_no_improve = 0
        # Save the best model
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  -> New best validation accuracy: {best_val_accuracy:.2f}%. Model saved.")
    else:
        epochs_no_improve += 1
    
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered after {patience} epochs with no improvement.")
        break

print("\n✅ Training finished.")


--- Starting Training with Augmentation and Early Stopping ---
Epoch [1/30] | Train Loss: 2.2971 | Train Acc: 52.65% | Val Acc: 68.82%
  -> New best validation accuracy: 68.82%. Model saved.
Epoch [2/30] | Train Loss: 1.0295 | Train Acc: 74.94% | Val Acc: 70.48%
  -> New best validation accuracy: 70.48%. Model saved.
Epoch [3/30] | Train Loss: 0.7045 | Train Acc: 82.04% | Val Acc: 72.22%
  -> New best validation accuracy: 72.22%. Model saved.
Epoch [4/30] | Train Loss: 0.5168 | Train Acc: 86.98% | Val Acc: 72.14%
Epoch [5/30] | Train Loss: 0.3849 | Train Acc: 90.51% | Val Acc: 74.63%
  -> New best validation accuracy: 74.63%. Model saved.
Epoch [6/30] | Train Loss: 0.2818 | Train Acc: 93.25% | Val Acc: 73.71%
Epoch [7/30] | Train Loss: 0.2181 | Train Acc: 95.06% | Val Acc: 72.80%
Epoch [8/30] | Train Loss: 0.1728 | Train Acc: 96.22% | Val Acc: 72.31%
Epoch [9/30] | Train Loss: 0.1574 | Train Acc: 96.49% | Val Acc: 72.39%
Epoch [10/30] | Train Loss: 0.1300 | Train Acc: 97.06% | Val Acc

In [30]:
save_path = './shared_resnet.pth'
        
# Save the model's state_dict
torch.save(model.state_dict(), save_path)

print(f"✅ Model saved to {save_path}")

✅ Model saved to ./shared_resnet.pth
